# 16 · Final test (run6: Juice Shop + DVWA, and the real tools)

This is the second locked test. run5 already tested the detector, the session layer and the attack type, and its
result stays as it is. run6 tests what was added after run5, with everything else in place:

- the **stage** of each attacker (recon, intrusion, exploitation, data theft) and the `critical` level,
- the **next step** of a multi-step attacker (Markov chain), on the campaign sessions,
- the **defence**: block / rate limit / allow per session, with the fix and the WAF rule,
- the **unusual-request** mark (Isolation Forest, review only),
- the same detection tables as notebook 09, because the session layer changed after run5 (scan on common pages).

It also runs the **8 real-tool logs** (sqlmap, nikto and hydra on both sites, and the stronger sqlmap run). They were
captured from the lab and sealed without being opened. In a tool log, every request from the tools container
(`172.19.0.7`) is an attack and every other IP is a normal visitor.

**Rules for this notebook:**
- The code is frozen first (tag `v3.0-final-test`), and the commit is printed below; it must say `(clean)`.
- With `RUN_FINAL = False` it runs on development data (run4 and camp7), to test the notebook itself. Some of that is
  in-sample (the next-step model and the block rules were trained on camp7), so the dev numbers mean nothing.
  The tool section runs on a cut of a dev log (one attacker IP plus the normal users), only to test the code.
- The SHA-256 of every file read is printed first. In the final run it is compared with `run6_hashes.txt`,
  `tools_hashes.txt` and `tools2_hashes.txt`; if any file changed, the notebook stops before any result is shown.
- `RUN_FINAL = True` is run **once**. Whatever comes out goes in the report. Nothing is changed after seeing it.

In [ ]:
RUN_FINAL = False   # set to True ONCE, after the code is frozen

In [ ]:
import hashlib, re, subprocess, sys, tempfile
from pathlib import Path, PureWindowsPath
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from eval.final_test import (TOOL_IP, defence_summary, investigate, next_step_summary, recall_by_type, run,
                             run_tool, stage_summary, summary, type_accuracy, unusual_summary)

commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', 'status', '--porcelain', '--', 'ml', 'eval', 'backend'], cwd=ROOT,
                       capture_output=True, text=True).stdout.strip()
print('code commit:', commit, '(uncommitted changes in ml/eval/backend!)' if dirty else '(clean)')

D = ROOT / 'data'
# name: (log, request labels, session labels or None)
if RUN_FINAL:
    RUNS = {'Juice Shop (run6)': (D / 'raw' / 'juiceshop_generated_run6.log', D / 'labels' / 'requests_run6.csv',
                                  D / 'labels' / 'sessions_run6.csv'),
            'DVWA (run6)': (D / 'raw' / 'dvwa_generated_run6.log', D / 'labels' / 'requests_dvwa_run6.csv',
                            D / 'labels' / 'sessions_dvwa_run6.csv')}
    TOOLS = {f'{tool} on {site}': D / 'raw' / f'tool_{tool}_{site}.log'
             for tool in ['sqlmap', 'sqlmap2', 'nikto', 'hydra'] for site in ['juiceshop', 'dvwa']}
else:
    RUNS = {'Juice Shop (run4, dev)': (D / 'raw' / 'juiceshop_generated_run4.log', D / 'labels' / 'requests_run4.csv', None),
            'DVWA (run4, dev)': (D / 'raw' / 'dvwa_generated_run4.log', D / 'labels' / 'requests_dvwa_run4.csv', None),
            'Juice Shop (camp7, dev)': (D / 'raw' / 'juiceshop_generated_camp7.log', D / 'labels' / 'requests_camp7.csv',
                                        D / 'labels' / 'sessions_camp7.csv'),
            'DVWA (camp7, dev)': (D / 'raw' / 'dvwa_generated_camp7.log', D / 'labels' / 'requests_dvwa_camp7.csv',
                                  D / 'labels' / 'sessions_dvwa_camp7.csv')}
    for name, files in list(RUNS.items()):
        if not all(f.exists() for f in files if f):
            print(f'{name}: skipped, files not on this machine')
            del RUNS[name]
    TOOLS = {}   # filled in section 7 from a dev log

# Fingerprint of every file read. In the final run it must match the hash lists written when the files were made.
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest().upper()   # upper: same as Get-FileHash

def read_hashes(path):
    raw = Path(path).read_bytes()   # missing: ask for it, never skip the check
    text = raw.decode('utf-16') if raw[:2] in (b'\xff\xfe', b'\xfe\xff') else raw.decode('utf-8-sig')
    return {PureWindowsPath(p.strip()).name: h.upper() for h, p in re.findall(r'\b([0-9A-Fa-f]{64})\s+(.+)', text)}

files = [f for fs in RUNS.values() for f in fs if f] + list(TOOLS.values())
hashes = {f.name: sha256(f) for f in files}
for file_name, h in hashes.items():
    print(h, file_name)
if RUN_FINAL:
    expected = {}
    for hash_list in ['run6_hashes.txt', 'tools_hashes.txt', 'tools2_hashes.txt']:
        expected.update(read_hashes(D / 'labels' / hash_list))
    changed = [f for f, h in hashes.items() if expected.get(f) != h]
    if changed:
        raise RuntimeError(f'Not as in the hash lists (changed, or missing from the lists): {changed}. '
                           'Stop here, do not look at any result.')
    print('all files match the hash lists')

results = {}
for name, (log, labels, _) in RUNS.items():
    req, sessions, report = run(log, labels)
    req, sessions = investigate(req, sessions)
    results[name] = (req, sessions)
    print(name, report)

## 1. Requests: how many attacks are caught, and how many false alarms

In [ ]:
pd.concat({name: summary(req) for name, (req, _) in results.items()})

## 2. Recall by attack type

In [ ]:
pd.concat({name: recall_by_type(req) for name, (req, _) in results.items()})

## 3. Sessions: is every attacker found, and no normal user flagged?

`critical` = high risk and the attacker reached exploitation or data theft.

In [ ]:
pd.concat({name: pd.crosstab(s['truth'], s['risk_level']) for name, (_, s) in results.items()})

## 4. Attack type of the caught payload attacks

In [ ]:
for name, (req, _) in results.items():
    table, acc = type_accuracy(req)
    print(f'{name}: {acc:.1%} correct')
    display(table)

## 5. Stage, next step and defence

**Stage**: the furthest stage each session reached. Normal users should have none.

**Next step**: only the campaign sessions (attackers that do several kinds of attack one after the other). *Order read exactly* = the steps read from the log are the steps the generator ran. Top-1 / top-3 = how often the real next step is the model's first guess / among its first three, on the true order and on the order read from the log. A random guess among the 6 steps is right 1 time in 6.

**Defence**: the right action comes from the labels: any payload attack -> block, only brute force, enumeration or scanning -> rate limit, normal -> allow. The two mistakes that matter are an attacker allowed and a normal user blocked.

In [ ]:
pd.concat({name: stage_summary(s) for name, (_, s) in results.items()})

In [ ]:
steps = {name: next_step_summary(req, RUNS[name][2]) for name, (req, _) in results.items() if RUNS[name][2]}
pd.DataFrame(steps)

In [ ]:
pd.DataFrame({name: defence_summary(req, s) for name, (req, s) in results.items()})

## 6. The unusual-request mark

Review only: it never changes a label or a risk. It marks the 1% most unusual requests of each log (Isolation Forest fitted on that log). The row that matters most is how many normal requests it marks, and whether it marks any attack the detector and the session layer missed.

In [ ]:
pd.DataFrame({name: unusual_summary(req) for name, (req, _) in results.items()})

## 7. Real tools

Each tool log is one run of one tool against one site. Requests from the tools container are attacks; any other IP
in the file (for example the browser on the host) is a normal visitor. Per file: how many tool requests are caught,
false alarms on the other visitors, the risk level and the action of the tool's session, the attack types given to
what was caught, and how many tool requests got the unusual mark.

In [ ]:
if not RUN_FINAL:
    # Code check only: a dev log cut down to the normal users plus one attacker IP, in the shape of a tool log.
    dev_log, dev_labels, _ = RUNS['DVWA (run4, dev)']
    lab = pd.read_csv(dev_labels, keep_default_na=False)
    normal_ips = set(lab['ip']) - set(lab.loc[lab['label'] != 'normal', 'ip'])
    attacker = lab.loc[lab['attack_type'] == 'sqli', 'ip'].value_counts().index[0]
    lines = [l for l in open(dev_log, encoding='utf-8') if l.split(' ', 1)[0] in normal_ips | {attacker}]
    tmp = Path(tempfile.mkdtemp()) / 'tool_dev_check.log'
    tmp.write_text(''.join(lines), encoding='utf-8')
    TOOLS = {'dev check (one sqli attacker on DVWA run4)': tmp}
    TOOL_ATTACKER = attacker
else:
    TOOL_ATTACKER = TOOL_IP

rows, tool_results = {}, {}
for name, path in TOOLS.items():
    req, sessions, report = run_tool(path, attacker_ip=TOOL_ATTACKER)
    req, sessions = investigate(req, sessions)
    tool_results[name] = (req, sessions)
    tool = req['ip'] == TOOL_ATTACKER
    tool_sessions = sessions[sessions['session'].isin(req.loc[tool, 'session'])]
    caught = req.loc[tool & (req['model + sessions'] == 1)]
    rows[name] = {**report,
                  'caught (model + sessions)': round(float(req.loc[tool, 'model + sessions'].mean()), 3) if tool.any() else None,
                  'caught (model)': round(float(req.loc[tool, 'model'].mean()), 3) if tool.any() else None,
                  'caught (rules + sessions)': round(float(req.loc[tool, 'rules + sessions'].mean()), 3) if tool.any() else None,
                  'false alarms on other visitors': int(req.loc[~tool, 'model + sessions'].sum()),
                  'tool sessions': len(tool_sessions),
                  'tool risk levels': ', '.join(sorted(set(tool_sessions['risk_level']))),
                  'tool actions': ', '.join(sorted(set(tool_sessions['action']))),
                  'tool stages': ', '.join(sorted(set(tool_sessions['stage'].replace('', 'none')))),
                  'types given': ', '.join(f'{t} {n}' for t, n in caught['type_pred'].replace('', 'session only')
                                           .value_counts().items()),
                  'tool requests marked unusual': int(req.loc[tool, 'unusual'].sum())}
pd.DataFrame(rows).T

## 8. Same logs without request bodies

Most real access logs do not store the body. This shows what is lost (attacks sent in a POST body) and checks that false alarms stay at zero.

In [ ]:
pd.concat({name: summary(run(log, labels, drop_body=True)[0]) for name, (log, labels, _) in RUNS.items()}
          | {name: summary(run_tool(path, attacker_ip=TOOL_ATTACKER, drop_body=True)[0]) for name, path in TOOLS.items()})

## 9. What this shows

*(Write this after the one final run, from the numbers above. Do not change the code or rerun.)*